# 1. Environment Setup

In [1]:
import os
os.environ['KAGGLE_USERNAME'] = "shubham710825"
os.environ['KAGGLE_KEY']      = "KGAT_e60d45e01b28496702dd6d2b0fd055ba"
print("Kaggle API configured")

import os, numpy as np, pandas as pd, librosa, gc
from tqdm import tqdm
import tensorflow as tf
import kagglehub
print("Libraries loaded")

Kaggle API configured
Libraries loaded


#Mount with drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#save on drive

In [3]:
SAVE_DIR = "/content/drive/MyDrive/deepfake_audio"

os.makedirs(SAVE_DIR, exist_ok=True)

print("Audio SAVE_DIR ready!")

Audio SAVE_DIR ready!


# 2. Reproducibility

In [4]:
import random

def set_seed(seed: int = 42) -> None:
    """Fix all random seeds for reproducible experiments."""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)
print("Seeds fixed → reproducible results")

Seeds fixed → reproducible results


# 3. Dataset Download

In [5]:
path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")
print(f"Dataset path: {path}")

100%|██████████| 23.6G/23.6G [04:24<00:00, 95.8MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1


# 4. Protocol Loading

In [6]:
la_path       = os.path.join(path, "LA", "LA")
protocol_path = os.path.join(la_path, "ASVspoof2019_LA_cm_protocols")
COLS          = ["speaker_id", "file_name", "unused", "system_id", "label"]

def load_protocol(filepath: str) -> pd.DataFrame:
    """Load an ASVspoof protocol file and encode labels (bonafide=0, spoof=1)."""
    df = pd.read_csv(filepath, sep=" ", header=None)
    df.columns = COLS
    df["label"] = df["label"].map({"bonafide": 0, "spoof": 1})
    return df

def add_file_paths(df: pd.DataFrame, folder: str) -> pd.DataFrame:
    """Attach full .flac file paths to each protocol row."""
    base = os.path.join(la_path, folder, "flac")
    df["file_path"] = df["file_name"].apply(
        lambda x: os.path.join(base, x + ".flac"))
    return df

df_train = load_protocol(os.path.join(protocol_path, "ASVspoof2019.LA.cm.train.trn.txt"))
df_dev   = load_protocol(os.path.join(protocol_path, "ASVspoof2019.LA.cm.dev.trl.txt"))
df_eval  = load_protocol(os.path.join(protocol_path, "ASVspoof2019.LA.cm.eval.trl.txt"))

df_train = add_file_paths(df_train, "ASVspoof2019_LA_train")
df_dev   = add_file_paths(df_dev,   "ASVspoof2019_LA_dev")
df_eval  = add_file_paths(df_eval,  "ASVspoof2019_LA_eval")

df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
df_dev   = df_dev.sample(frac=1,   random_state=42).reset_index(drop=True)
df_eval  = df_eval.sample(frac=1,  random_state=42).reset_index(drop=True)

print(f"Train: {len(df_train):,}  |  Dev: {len(df_dev):,}  |  Eval: {len(df_eval):,}")

Train: 25,380  |  Dev: 24,844  |  Eval: 71,237


# 5. Audio Feature Extraction

In [7]:
TARGET_SR = 16000
MAX_LEN   = 4 * TARGET_SR
MAX_PAD   = 128
N_MELS    = 64

def extract_logmel(audio: np.ndarray, sr: int = TARGET_SR) -> np.ndarray:
    mel    = librosa.feature.melspectrogram(
                 y=audio, sr=sr, n_fft=1024, hop_length=512, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel)
    log_mel = (log_mel - np.mean(log_mel)) / (np.std(log_mel) + 1e-6)

    if log_mel.shape[1] < MAX_PAD:
        log_mel = np.pad(log_mel, ((0, 0), (0, MAX_PAD - log_mel.shape[1])))
    else:
        log_mel = log_mel[:, :MAX_PAD]

    return log_mel  # (64, 128)


def augment_audio(audio):

    # =========================
    # Random Noise
    # =========================

    if np.random.rand() < 0.5:

        noise = np.random.normal(
            0,
            0.003,
            len(audio)
        )

        audio = audio + noise

    # =========================
    # Volume Perturbation
    # =========================

    if np.random.rand() < 0.5:

        gain = np.random.uniform(
            0.8,
            1.2
        )

        audio = audio * gain

    # =========================
    # Time Shift
    # =========================

    if np.random.rand() < 0.5:

        shift = np.random.randint(
            int(0.05 * len(audio))
        )

        audio = np.roll(audio, shift)

    # =========================
    # Mild Reverb Simulation
    # =========================

    if np.random.rand() < 0.3:

        echo = np.pad(
            audio[:-4000],
            (4000, 0),
            mode='constant'
        ) * 0.3

        audio = audio + echo

    # =========================
    # Normalize
    # =========================

    if np.max(np.abs(audio)) > 0:

        audio = audio / np.max(np.abs(audio))

    return audio


def load_and_extract(df: pd.DataFrame,
                     limit: int   = None,
                     augment: bool = False):
    X, y = [], []
    n = min(len(df), limit) if limit else len(df)

    for i in tqdm(range(n), desc="Loading audio"):
        try:
            audio, _ = librosa.load(df.iloc[i]["file_path"], sr=TARGET_SR)

            # peak-normalize
            if np.max(np.abs(audio)) > 0:
                audio /= np.max(np.abs(audio))

            # fixed-length clip
            audio = audio[:MAX_LEN] if len(audio) > MAX_LEN else \
                    np.pad(audio, (0, MAX_LEN - len(audio)))

            X.append(np.expand_dims(extract_logmel(audio), -1))
            y.append(df.iloc[i]["label"])

            # augment only bonafide to address class imbalance
            if augment and df.iloc[i]["label"] == 0:
                aug = augment_audio(audio)
                X.append(np.expand_dims(extract_logmel(aug), -1))
                y.append(0)

            del audio
        except Exception:
            continue

    return np.array(X, dtype=np.float32), np.array(y)

print("Feature functions ready")

Feature functions ready


#  6. Load Full Dataset

In [8]:
print("\n[1/3] Loading training set…")
X_train_raw, y_train_raw = load_and_extract(df_train, augment=True)
gc.collect()
print(f"      Shape: {X_train_raw.shape}  |  Labels: {np.bincount(y_train_raw)}")

print("\n[2/3] Loading validation set…")
X_val, y_val = load_and_extract(df_dev)
gc.collect()
print(f"      Shape: {X_val.shape}  |  Labels: {np.bincount(y_val)}")

print("\n[3/3] Loading evaluation set…")
X_test, y_test = load_and_extract(df_eval)
gc.collect()
print(f"      Shape: {X_test.shape}  |  Labels: {np.bincount(y_test)}")


[1/3] Loading training set…


Loading audio: 100%|██████████| 25380/25380 [04:35<00:00, 92.15it/s]


      Shape: (27960, 64, 128, 1)  |  Labels: [ 5160 22800]

[2/3] Loading validation set…


Loading audio: 100%|██████████| 24844/24844 [03:48<00:00, 108.82it/s]


      Shape: (24844, 64, 128, 1)  |  Labels: [ 2548 22296]

[3/3] Loading evaluation set…


Loading audio: 100%|██████████| 71237/71237 [10:53<00:00, 109.01it/s]


      Shape: (71237, 64, 128, 1)  |  Labels: [ 7355 63882]


# 7. Class Balancing (Train Only)

In [9]:
bon = np.where(y_train_raw == 0)[0]
spo = np.where(y_train_raw == 1)[0]
print(f"\nBefore balance → Bonafide: {len(bon):,}  |  Spoof: {len(spo):,}")

np.random.seed(42)
idx = np.concatenate([bon, spo])
np.random.shuffle(idx)

X_train = X_train_raw[idx]
y_train = y_train_raw[idx]
del X_train_raw, y_train_raw
gc.collect()

print(f"After  balance → {np.bincount(y_train)}  |  Train shape: {X_train.shape}")


Before balance → Bonafide: 5,160  |  Spoof: 22,800
After  balance → [ 5160 22800]  |  Train shape: (27960, 64, 128, 1)


# 8. Model Architecture

In [10]:
from tensorflow.keras import models, layers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC

FEATURE_DIM   = 128
INPUT_SHAPE   = (64, 128, 1)

def build_full_model(input_shape: tuple = INPUT_SHAPE) -> tf.keras.Model:

    model = models.Sequential([
        layers.Input(shape=input_shape),

        # ── Convolutional blocks ──────────────────────────
        layers.Conv2D(32,  (3, 3), activation='relu', padding='same', name='conv1'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64,  (3, 3), activation='relu', padding='same', name='conv2'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.30),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv4'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.30),

        # ── Audio Feature Vector (128-dim) ────────────────
        layers.Dense(FEATURE_DIM, activation='relu', name='audio_feature_vector'),
        layers.Dropout(0.40),

        # ── Classification head ───────────────────────────
        layers.Dense(1, activation='sigmoid', name='classifier_head'),
    ], name="AudioDeepfakeDetector")

    return model


model = build_full_model()
model.compile(
    optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
    loss='binary_crossentropy',
    metrics=['accuracy', AUC(name='auc')]
)
model.summary()
print(f"\n Model ready  |  Feature vector layer: 'audio_feature_vector' ({FEATURE_DIM}-dim)")

Model: "AudioDeepfakeDetector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 64, 128, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 32, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 16, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 16, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 16, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 8, 16, 128)     │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ audio_feature_vector (Dense)    │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier_head (Dense)         │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,897 (1003.50 KB)

 Trainable params: 256,897 (1003.50 KB)

 Non-trainable params: 0 (0.00 B)


 Model ready  |  Feature vector layer: 'audio_feature_vector' (128-dim)


# 9. Callbacks & Class Weights

In [11]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)
checkpoint = ModelCheckpoint(
    os.path.join(
        SAVE_DIR,
        "best_audio_model.keras"
    ),
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-7,
    verbose=1
)

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(cw))
print(f"Class weights: {class_weight}")

Class weights: {0: np.float64(2.7093023255813953), 1: np.float64(0.6131578947368421)}


# 10. Training

In [12]:
print("\n" + "="*55)
print("  TRAINING AUDIO MODULE")
print("="*55)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop, checkpoint, lr_reducer],
    class_weight=class_weight
)


  TRAINING AUDIO MODULE
Epoch 1/50
874/874 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5375 - auc: 0.5264 - loss: 0.6897
Epoch 1: val_loss improved from None to 0.89870, saving model to /content/drive/MyDrive/deepfake_audio/best_audio_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/deepfake_audio/best_audio_model.keras
874/874 ━━━━━━━━━━━━━━━━━━━━ 34s 27ms/step - accuracy: 0.5664 - auc: 0.6008 - loss: 0.6683 - val_accuracy: 0.3520 - val_auc: 0.7788 - val_loss: 0.8987 - learning_rate: 1.0000e-04
Epoch 2/50
872/874 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7579 - auc: 0.8332 - loss: 0.5007
Epoch 2: val_loss did not improve from 0.89870
874/874 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.7713 - auc: 0.8522 - loss: 0.4745 - val_accuracy: 0.4698 - val_auc: 0.8577 - val_loss: 0.9354 - learning_rate: 1.0000e-04
Epoch 3/50
870/874 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8135 - auc: 0.8974 - loss: 0.4002
Epoch 3: val_loss improved from 0.89870 to 0.

# 11. Evaluation

In [13]:
from sklearn.metrics import (classification_report,
    confusion_matrix, accuracy_score, f1_score)


bon_test   = np.where(y_test == 0)[0]
spo_test   = np.where(y_test == 1)[0]
spo_test_s = np.random.choice(spo_test, size=len(bon_test), replace=False)
test_idx   = np.concatenate([bon_test, spo_test_s])
np.random.shuffle(test_idx)
X_test_bal = X_test[test_idx]
y_test_bal = y_test[test_idx]

print(f"\nBalanced test set: {np.bincount(y_test_bal)}")

y_val_prob  = model.predict(X_val,       verbose=0).flatten()
y_test_prob = model.predict(X_test_bal,  verbose=0).flatten()


print("\n--- Threshold Search (val) ---")
best_f1, best_thresh = 0, 0.5
for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    yp  = (y_val_prob > t).astype(int)
    f1  = f1_score(y_val, yp, average='macro')
    acc = accuracy_score(y_val, yp)
    print(f"  Thresh {t:.2f} → F1: {f1:.4f}  |  Acc: {acc:.4f}")
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print(f"\n  Best Threshold: {best_thresh}")
y_pred = (y_test_prob > best_thresh).astype(int)

print("\n" + "="*55)
print("  FINAL TEST RESULTS")
print("="*55)
print(f"  Accuracy : {accuracy_score(y_test_bal, y_pred):.4f}")
print(f"  F1 Macro : {f1_score(y_test_bal, y_pred, average='macro'):.4f}")
print()
print(classification_report(y_test_bal, y_pred, target_names=['bonafide', 'spoof']))
print(confusion_matrix(y_test_bal, y_pred))


Balanced test set: [7355 7355]

--- Threshold Search (val) ---
  Thresh 0.30 → F1: 0.9162  |  Acc: 0.9721
  Thresh 0.35 → F1: 0.9205  |  Acc: 0.9732
  Thresh 0.40 → F1: 0.9243  |  Acc: 0.9742
  Thresh 0.45 → F1: 0.9268  |  Acc: 0.9748
  Thresh 0.50 → F1: 0.9308  |  Acc: 0.9758
  Thresh 0.55 → F1: 0.9321  |  Acc: 0.9761
  Thresh 0.60 → F1: 0.9333  |  Acc: 0.9762

  Best Threshold: 0.6

  FINAL TEST RESULTS
  Accuracy : 0.8717
  F1 Macro : 0.8715

              precision    recall  f1-score   support

    bonafide       0.84      0.91      0.88      7355
       spoof       0.91      0.83      0.87      7355

    accuracy                           0.87     14710
   macro avg       0.87      0.87      0.87     14710
weighted avg       0.87      0.87      0.87     14710

[[6726  629]
 [1258 6097]]


In [14]:
print("=" * 40)
print("DATASET SAMPLE SIZE SUMMARY")
print("=" * 40)

print(f"\nTraining Set:")
print(f"  Total samples : {len(X_train)}")
print(f"  Bonafide (0)  : {np.sum(y_train == 0)}")
print(f"  Spoof    (1)  : {np.sum(y_train == 1)}")

print(f"\nValidation Set:")
print(f"  Total samples : {len(X_val)}")
print(f"  Bonafide (0)  : {np.sum(y_val == 0)}")
print(f"  Spoof    (1)  : {np.sum(y_val == 1)}")

print(f"\nTest Set (Balanced):")
print(f"  Total samples : {len(X_test_bal)}")
print(f"  Bonafide (0)  : {np.sum(y_test_bal == 0)}")
print(f"  Spoof    (1)  : {np.sum(y_test_bal == 1)}")

print(f"\nGrand Total:")
total = len(X_train) + len(X_val) + len(X_test_bal)
print(f"  Total samples used : {total}")

print(f"\nFeature Shape:")
print(f"  X_train shape : {X_train.shape}")
print(f"  X_val shape   : {X_val.shape}")
print(f"  X_test shape  : {X_test_bal.shape}")
print("=" * 40)



DATASET SAMPLE SIZE SUMMARY

Training Set:
  Total samples : 27960
  Bonafide (0)  : 5160
  Spoof    (1)  : 22800

Validation Set:
  Total samples : 24844
  Bonafide (0)  : 2548
  Spoof    (1)  : 22296

Test Set (Balanced):
  Total samples : 14710
  Bonafide (0)  : 7355
  Spoof    (1)  : 7355

Grand Total:
  Total samples used : 67514

Feature Shape:
  X_train shape : (27960, 64, 128, 1)
  X_val shape   : (24844, 64, 128, 1)
  X_test shape  : (14710, 64, 128, 1)


#SAMPLE REAL/FAKE LIST

In [15]:
print("\nREAL AUDIO SAMPLES\n")

real_samples = df_eval[
    df_eval["label"] == 0
].head(6)

for i, row in real_samples.iterrows():

    print(
        f"[REAL] {row['file_path']}"
    )


print("\nFAKE AUDIO SAMPLES\n")

fake_samples = df_eval[
    df_eval["label"] == 1
].head(6)

for i, row in fake_samples.iterrows():

    print(
        f"[FAKE] {row['file_path']}"
    )


REAL AUDIO SAMPLES

[REAL] /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA/ASVspoof2019_LA_eval/flac/LA_E_6465123.flac
[REAL] /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA/ASVspoof2019_LA_eval/flac/LA_E_4202258.flac
[REAL] /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA/ASVspoof2019_LA_eval/flac/LA_E_7310993.flac
[REAL] /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA/ASVspoof2019_LA_eval/flac/LA_E_1145965.flac
[REAL] /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA/ASVspoof2019_LA_eval/flac/LA_E_4785445.flac
[REAL] /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA/ASVspoof2019_LA_eval/flac/LA_E_3268391.flac

FAKE AUDIO SAMPLES

[FAKE] /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA/ASVspoof2019_LA_eval/flac/LA_E_9509114.flac
[FAKE] /root/.cache/kagglehub/datasets/awsaf49/asvpoof-

#SAVE SAMPLE LIST TO TXT


In [16]:
sample_txt = os.path.join(
    SAVE_DIR,
    "audio_test_samples.txt"
)

with open(sample_txt, "w") as f:

    f.write("REAL AUDIO SAMPLES\n\n")

    for _, row in real_samples.iterrows():

        f.write(
            f"[REAL] {row['file_path']}\n"
        )

    f.write("\n\nFAKE AUDIO SAMPLES\n\n")

    for _, row in fake_samples.iterrows():

        f.write(
            f"[FAKE] {row['file_path']}\n"
        )

print(
    f"Saved sample list → {sample_txt}"
)

Saved sample list → /content/drive/MyDrive/deepfake_audio/audio_test_samples.txt


# 12. Audio Feature Vector Extractor (FOR MULTIMODAL FUSION)

In [17]:
def build_feature_extractor(trained_model):
    dummy = np.zeros((1, 64, 128, 1), dtype=np.float32)
    trained_model(dummy)

    return tf.keras.Model(
        inputs  = trained_model.inputs,
        outputs = trained_model.get_layer("audio_feature_vector").output,
        name    = "AudioFeatureExtractor"
    )


audio_feature_extractor = build_feature_extractor(model)
audio_feature_extractor.summary()

audio_feature_extractor.save(
    os.path.join(
        SAVE_DIR,
        "audio_feature_extractor.keras"
    )
)
print("\n Saved: audio_feature_extractor.keras  (share with teammates)")


def get_audio_feature_vector(audio_path: str) -> np.ndarray:

    audio, _ = librosa.load(audio_path, sr=TARGET_SR)

    if np.max(np.abs(audio)) > 0:
        audio /= np.max(np.abs(audio))

    audio = audio[:MAX_LEN] if len(audio) > MAX_LEN else \
            np.pad(audio, (0, MAX_LEN - len(audio)))

    logmel = extract_logmel(audio)
    logmel = logmel[np.newaxis, :, :, np.newaxis]

    vector = audio_feature_extractor.predict(logmel, verbose=0)
    return vector.flatten()


def batch_extract_audio_vectors(df: pd.DataFrame,
                                 batch_size: int = 64) -> np.ndarray:

    all_vectors = []

    for i in tqdm(range(0, len(df), batch_size), desc="Extracting audio vectors"):
        batch_df   = df.iloc[i : i + batch_size]
        batch_X    = []

        for _, row in batch_df.iterrows():
            try:
                audio, _ = librosa.load(row["file_path"], sr=TARGET_SR)
                if np.max(np.abs(audio)) > 0:
                    audio /= np.max(np.abs(audio))
                audio = audio[:MAX_LEN] if len(audio) > MAX_LEN else \
                        np.pad(audio, (0, MAX_LEN - len(audio)))
                batch_X.append(np.expand_dims(extract_logmel(audio), -1))
            except Exception:
                batch_X.append(np.zeros((N_MELS, MAX_PAD, 1)))

        batch_X = np.array(batch_X, dtype=np.float32)
        vecs    = audio_feature_extractor.predict(batch_X, verbose=0)
        all_vectors.append(vecs)

    return np.vstack(all_vectors)

print("Feature extractor functions defined")

Model: "AudioFeatureExtractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 128, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 64, 128, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 32, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 16, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 16, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 16, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 8, 16, 128)     │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ audio_feature_vector (Dense)    │ (None, 128)            │        16,512 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,768 (1003.00 KB)

 Trainable params: 256,768 (1003.00 KB)

 Non-trainable params: 0 (0.00 B)


 Saved: audio_feature_extractor.keras  (share with teammates)
Feature extractor functions defined


 # 13. Demo — Feature Vector Output

In [18]:
print("\n" + "="*55)
print("  AUDIO FEATURE VECTOR — DEMO OUTPUT")
print("="*55)

sample_df      = df_eval.sample(5, random_state=42).reset_index(drop=True)
sample_vectors = batch_extract_audio_vectors(sample_df)

print(f"\n  Input  : audio waveform (.flac)")
print(f"  Output : feature vector — shape {sample_vectors.shape}")
print(f"\n  Sample vectors (5 clips):")

for i, (vec, label) in enumerate(zip(sample_vectors, sample_df["label"])):
    tag   = "bonafide" if label == 0 else "spoof"
    norm  = np.linalg.norm(vec)
    print(f"  [{i+1}] {tag:9s}  |  L2-norm: {norm:.3f}  |  "
          f"mean: {vec.mean():.4f}  |  std: {vec.std():.4f}")

print(f"""
  ─────────────────────────────────────────────
  FUSION MODULE INTEGRATION (for teammate)
  ─────────────────────────────────────────────
  Audio vector shape  : ({FEATURE_DIM},)
  Dtype               : float32
  Value range         : ≥ 0  (ReLU activation)
  Normalization       : L2-normalize before concat

  Example fusion code:
  ─────────────────────────────────────────────
  import numpy as np

  audio_vec = get_audio_feature_vector("file.flac")  # (128,)
  audio_vec = audio_vec / (np.linalg.norm(audio_vec) + 1e-6)

  # concat with image + video vectors
  multimodal_vec = np.concatenate([img_vec, video_vec, audio_vec])
  # → shape: (img_dim + vid_dim + 128,)
  ─────────────────────────────────────────────
""")


  AUDIO FEATURE VECTOR — DEMO OUTPUT


Extracting audio vectors: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]


  Input  : audio waveform (.flac)
  Output : feature vector — shape (5, 128)

  Sample vectors (5 clips):
  [1] spoof      |  L2-norm: 12.487  |  mean: 0.6544  |  std: 0.8888
  [2] spoof      |  L2-norm: 23.528  |  mean: 1.2081  |  std: 1.6927
  [3] spoof      |  L2-norm: 14.335  |  mean: 0.7354  |  std: 1.0318
  [4] spoof      |  L2-norm: 11.645  |  mean: 0.5972  |  std: 0.8383
  [5] bonafide   |  L2-norm: 14.170  |  mean: 0.6939  |  std: 1.0427

  ─────────────────────────────────────────────
  FUSION MODULE INTEGRATION (for teammate)
  ─────────────────────────────────────────────
  Audio vector shape  : (128,)
  Dtype               : float32
  Value range         : ≥ 0  (ReLU activation)
  Normalization       : L2-normalize before concat

  Example fusion code:
  ─────────────────────────────────────────────
  import numpy as np

  audio_vec = get_audio_feature_vector("file.flac")  # (128,)
  audio_vec = audio_vec / (np.linalg.norm(audio_vec) + 1e-6)

  # concat with image + video

# 14. Extract & Save All Vectors

In [19]:
print("Extracting audio feature vectors for all splits…")

audio_vecs_train = batch_extract_audio_vectors(df_train)
audio_vecs_val   = batch_extract_audio_vectors(df_dev)
audio_vecs_eval  = batch_extract_audio_vectors(df_eval)

np.save(
    os.path.join(
        SAVE_DIR,
        "audio_vectors_train.npy"
    ),
    audio_vecs_train
)

np.save(
    os.path.join(
        SAVE_DIR,
        "audio_vectors_val.npy"
    ),
    audio_vecs_val
)

np.save(
    os.path.join(
        SAVE_DIR,
        "audio_vectors_eval.npy"
    ),
    audio_vecs_eval
)
np.save(
    os.path.join(
        SAVE_DIR,
        "audio_labels_train.npy"
    ),
    y_train
)

np.save(
    os.path.join(
        SAVE_DIR,
        "audio_labels_val.npy"
    ),
    y_val
)

np.save(
    os.path.join(
        SAVE_DIR,
        "audio_labels_eval.npy"
    ),
    y_test
)

print(f"""
   Saved audio feature vectors:
   audio_vectors_train.npy  — shape {audio_vecs_train.shape}
   audio_vectors_val.npy    — shape {audio_vecs_val.shape}
   audio_vectors_eval.npy   — shape {audio_vecs_eval.shape}

   Teammate loads with:
   audio_vecs = np.load("audio_vectors_train.npy")
""")

Extracting audio feature vectors for all splits…


Extracting audio vectors: 100%|██████████| 1114/1114 [13:20<00:00,  1.39it/s]



   Saved audio feature vectors:
   audio_vectors_train.npy  — shape (25380, 128)
   audio_vectors_val.npy    — shape (24844, 128)
   audio_vectors_eval.npy   — shape (71237, 128)

   Teammate loads with:
   audio_vecs = np.load("audio_vectors_train.npy")



#15. Files Download

In [21]:
from google.colab import files
files.download(
    os.path.join(
        SAVE_DIR,
        "audio_feature_extractor.keras"
    )
)

files.download(
    os.path.join(
        SAVE_DIR,
        "best_audio_model.keras"
    )
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#test by your

In [46]:
print("[FINAL TEST] Upload audio for spoof detection...")

from google.colab import files
uploaded = files.upload()

uploaded_path = list(uploaded.keys())[0]

print(f"\nUploaded file: {uploaded_path}")

# =========================
# LOAD AUDIO
# =========================

audio, sr = librosa.load(
    uploaded_path,
    sr=TARGET_SR
)

# Peak normalize
if np.max(np.abs(audio)) > 0:
    audio = audio / np.max(np.abs(audio))

# Fixed length
audio = (
    audio[:MAX_LEN]
    if len(audio) > MAX_LEN
    else np.pad(audio, (0, MAX_LEN - len(audio)))
)

# =========================
# FEATURE EXTRACTION
# =========================

mel = librosa.feature.melspectrogram(
    y=audio,
    sr=sr,
    n_fft=1024,
    hop_length=512,
    n_mels=N_MELS
)

mel_db = librosa.power_to_db(
    mel,
    ref=np.max
)

# Normalize
mel_db = (
    mel_db - mel_db.mean()
) / (
    mel_db.std() + 1e-6
)

# Resize / pad to (64, 128)

if mel_db.shape[1] < MAX_PAD:

    pad_width = MAX_PAD - mel_db.shape[1]

    mel_db = np.pad(
        mel_db,
        ((0, 0), (0, pad_width)),
        mode='constant'
    )

else:

    mel_db = mel_db[:, :MAX_PAD]

# =========================
# MODEL INPUT SHAPE
# =========================

x = np.expand_dims(mel_db, axis=-1)
x = np.expand_dims(x, axis=0)

# =========================
# PREDICTION
# =========================

pred = model.predict(x, verbose=0)[0][0]

label = "FAKE" if pred >= best_thresh else "REAL"

confidence = (
    pred if label == "FAKE"
    else 1 - pred
)

# =========================
# OUTPUT
# =========================

print("\n==============================")
print(f"Prediction : {label}")
print(f"Confidence : {confidence*100:.2f}%")
print(f"Raw Score  : {pred:.4f}")
print("==============================")

[FINAL TEST] Upload audio for spoof detection...


Saving LA_E_9509114.flac to LA_E_9509114 (1).flac

Uploaded file: LA_E_9509114 (1).flac

Prediction : FAKE
Confidence : 100.00%
Raw Score  : 1.0000


In [23]:
gc.collect()

print("Notebook completed successfully!")

Notebook completed successfully!
